# AI Memory Semantic Block System

A Gemini assistant with long-term memory, using `semantic_memory.py` from this folder.

- Before each reply, it searches past memories by meaning (embeddings + ChromaDB) and adds the relevant ones, plus the last few messages, to the prompt.
- Your messages (except questions) are saved as **episodic** memories, which expire after 90 days unless you repeat them.
- Lasting facts saved with `assistant.remember(...)` are **semantic** memories, which stay until deleted.
- A new fact that contradicts a very similar old one replaces it (the old one can be restored for 7 days).
- Memories are saved in the `memory/` folder, so they are still there next time you open the notebook.

In [ ]:
# Cell 1: Install packages (only needed once per environment)
%pip install -q chromadb sentence-transformers tiktoken numpy google-genai

In [ ]:
# Cell 2: Imports and Gemini API key
import getpass
from google import genai
from semantic_memory import MemorySession, NLIContradictionChecker, SemanticMemoryStore, metrics

client = genai.Client(api_key=getpass.getpass("Enter your Gemini API key: "))
print("✅ Gemini client ready!")

In [ ]:
# Cell 3: Choose a model and check it responds
MODEL_NAME = "gemini-3-flash-preview"  # or "gemini-2.5-flash"

hello = client.models.generate_content(model=MODEL_NAME, contents="Say hello in one word")
print(f"✅ {MODEL_NAME} says: {hello.text.strip()}")

In [ ]:
# Cell 4: Open the memory store (downloads the embedding and contradiction models the first time)
store = SemanticMemoryStore(storage_path="memory", contradiction_checker=NLIContradictionChecker())
print("✅ Memory store ready:", store.stats("alice"))

In [ ]:
# Cell 5: Gemini assistant with semantic memory
class GeminiAssistant:
    def __init__(self, store, user_id, client, model_name, reply_chars=500):
        self.store = store
        self.session = MemorySession(user_id, store)
        self.client = client
        self.model_name = model_name
        self.reply_chars = reply_chars  # how much of each reply to keep as recent context

    def remember(self, fact, importance=1.0):
        """Save a lasting fact about the user (semantic memory, never expires)."""
        mem = self.session.add(fact, importance=importance, memory_type="semantic")
        print(f"🧠 Remembered [{mem.topic}] {mem.content}")
        self._show_replaced(mem)
        return mem

    @staticmethod
    def _show_replaced(mem):
        for old_id, old_text in mem.replaced:
            print(f"   ↪ Replaced: {old_text}  (undo: assistant.session.restore(['{old_id}']))")

    def chat(self, message, show_context=True):
        topic = self.store.classifier.classify(message)
        # Search before saving the message, so it doesn't find itself.
        context = self.session.build_context(message)
        # Questions aren't facts about the user, so only statements are saved long-term.
        # Every message still goes into the recent conversation.
        if not message.rstrip().endswith("?"):
            mem = self.session.add(message, topic=topic, importance=0.5, memory_type="episodic")
            self._show_replaced(mem)
        self.session.add_turn(f"User: {message}")

        print(f"👤 You: {message}")
        print(f"📌 Topic: {topic}")
        if show_context:
            print(context or "(no relevant memories)")

        prompt = f"""You are a helpful assistant with long-term memory of this user.
Use the memories below only when they are relevant to the message, and don't mention ones that aren't.

{context or "(no relevant memories)"}

User: {message}
Assistant:"""
        try:
            reply = self.client.models.generate_content(model=self.model_name, contents=prompt).text
        except Exception as exc:
            print(f"❌ Error: {exc}")
            return None
        self.session.add_turn(f"Assistant: {reply[:self.reply_chars]}")
        print(f"\n🤖 AI: {reply}")
        return reply

assistant = GeminiAssistant(store, "alice", client, MODEL_NAME)

In [ ]:
# Cell 6: Save some lasting facts (safe to re-run: repeats refresh instead of duplicating)
assistant.remember("I love my Rolex Submariner")
assistant.remember("I prefer analog watches over smartwatches")
assistant.remember("I am allergic to peanuts")
assistant.remember("Neapolitan pizza is my favourite food")

In [ ]:
# Cell 7: Chat
assistant.chat("Which watch should I buy next with a medium budget?")

In [ ]:
# Cell 8: A different topic only brings in related memories
assistant.chat("Suggest a dinner recipe for tonight")

In [ ]:
# Cell 9: An unrelated question gets no memories, just the recent messages
assistant.chat("What's the capital of France?")

In [ ]:
# Cell 10: Correct a fact - the contradicted memory is replaced (and can be restored)
assistant.remember("I am not allergic to peanuts")
assistant.chat("Suggest a dinner recipe for tonight")

In [ ]:
# Cell 11: Look inside the memory
print(assistant.session.stats(refresh=True))
for mem in assistant.session.search("watches", k=5):
    print(f"- [{mem.topic}:{mem.memory_type}] {mem.content}")
print(metrics.summary()["counters"])

In [ ]:
# Cell 12: Maintenance
# Remove expired episodic memories and deleted memories past their 7-day grace period:
print("Purged:", assistant.session.purge_expired())

# To delete a memory (restorable for 7 days with assistant.session.restore([...])):
# ids = [m.id for m in assistant.session.search("pizza", k=1)]
# assistant.session.delete(ids)